# Exploration: does this business have a CLV problem worth modelling?

Before fitting anything, two questions decide whether a lifetime-value model is the right
tool: do customers come back at all, and does the data let you see them doing it.

Real Online Retail II data throughout. Nothing here is simulated.

In [1]:
import sys
sys.path.insert(0, "src")

import pandas as pd

from clv.cohorts import cohort_retention, retention_matrix
from clv.data import transactions
from clv.rfm import summary

orders = transactions()
print(f"{len(orders):,} purchase occasions, {orders['customer_id'].nunique():,} customers")
print(f"{orders['invoice_date'].min().date()} to {orders['invoice_date'].max().date()}")

33,107 purchase occasions, 5,878 customers
2009-12-01 to 2011-12-09


## 1. One invoice is not one purchase

A single shopping session is often split across several invoice documents. BG/NBD assumes
at most one transaction per customer per time unit, so counting invoices inflates frequency.

In [2]:
multi = orders[orders["invoices"] > 1]
print(f"customer-days carrying more than one invoice: {len(multi):,} of {len(orders):,} "
      f"({len(multi)/len(orders):.1%})")
print(f"invoices in total: {orders['invoices'].sum():,}")
print(f"purchase occasions: {len(orders):,}")
print(f"\ncounting invoices would inflate observed frequency by "
      f"{orders['invoices'].sum()/len(orders) - 1:.1%}")

customer-days carrying more than one invoice: 3,008 of 33,107 (9.1%)
invoices in total: 36,969
purchase occasions: 33,107

counting invoices would inflate observed frequency by 11.7%


## 2. Do customers come back?

In [3]:
features = summary(orders)
print(f"customers acquired in the calibration window: {len(features):,}")
print(f"repeat buyers (frequency > 0):                {(features['frequency'] > 0).mean():.1%}")
print(f"active in the holdout window:                 {(features['holdout_revenue'] > 0).mean():.1%}")
print()
print(features[["frequency", "recency", "T", "monetary_value"]].describe().round(1).to_string())

customers acquired in the calibration window: 4,969
repeat buyers (frequency > 0):                67.3%
active in the holdout window:                 51.9%

       frequency  recency       T  monetary_value
count     4969.0   4969.0  4969.0          4969.0
mean         3.7    192.7   363.6           280.9
std          7.9    188.7   157.0           534.5
min          0.0      0.0     0.0             0.0
25%          0.0      0.0   239.0             0.0
50%          1.0    152.0   400.0           200.5
75%          4.0    362.0   499.0           371.0
max        172.0    555.0   555.0         21535.9


**67% buy more than once.** That is what makes a CLV model worth fitting here.

The contrast matters: on the Olist marketplace in
[unit-economics-olist](https://github.com/arielabade/unit-economics-olist), 97% of customers
buy exactly once. There, fitting BG/NBD would be wasted effort — there is no repeat process
to model, and the correct business conclusion is that CAC must clear on the first order.

Choosing the right dataset for the question is part of the work.

## 3. Retention settles rather than decaying to zero

In [4]:
matrix = retention_matrix(cohort_retention(orders))
average = (matrix.mean() * 100).head(13).round(1)
print("average retention by months since acquisition (%)")
print(average.to_string())

average retention by months since acquisition (%)
month_index
0     100.0
1      21.2
2      21.9
3      21.6
4      20.5
5      18.8
6      17.8
7      17.7
8      15.6
9      15.3
10     15.0
11     16.4
12     18.2


The first-month drop is steep, then cohorts stabilise around 15–20% monthly. A business
that decayed to zero would not need a model; one that stabilises has a lifetime worth
estimating.

## 4. The RFM conventions that are easy to invert

In [5]:
example = features.sort_values("frequency", ascending=False).head(3)
print(example[["customer_id", "frequency", "recency", "T", "monetary_value"]].to_string(index=False))
print()
print("frequency  repeat purchases, so a one-time buyer is 0, not 1")
print("recency    AGE AT LAST PURCHASE (first-to-last gap), not days since")
print("T          first purchase to the end of the calibration window")
print()
print("Inverting recency silently flips the model's notion of who is still alive:")
print("no error, just wrong answers. There is a test pinning it.")

 customer_id  frequency  recency   T  monetary_value
       14911        172      555 555     1132.579186
       15311        149      553 555      596.686913
       14606        139      553 553      184.139856

frequency  repeat purchases, so a one-time buyer is 0, not 1
recency    AGE AT LAST PURCHASE (first-to-last gap), not days since
T          first purchase to the end of the calibration window

Inverting recency silently flips the model's notion of who is still alive:
no error, just wrong answers. There is a test pinning it.


## 5. What the fitted models produced

In [6]:
import json
metrics = json.load(open("reports/metrics.json"))
comparison = pd.DataFrame({
    "BG/NBD + Gamma-Gamma": metrics["bgnbd"],
    "LightGBM": metrics["lightgbm"],
}).T.round(3)
print(comparison.to_string())
print()
deciles = pd.read_csv("reports/deciles_bgnbd.csv")
lgbm = pd.read_csv("reports/deciles_lightgbm.csv")
side = pd.DataFrame({
    "decile": deciles["decile"],
    "bgnbd_lift": deciles["lift_vs_average"].round(2),
    "lightgbm_lift": lgbm["lift_vs_average"].round(2),
})
print(side.to_string(index=False))

                          mae     rmse     bias  spearman
BG/NBD + Gamma-Gamma  484.074  1126.86  -18.016     0.601
LightGBM              645.024  1789.76  139.800     0.485

 decile  bgnbd_lift  lightgbm_lift
      1        5.09           4.26
      2        1.67           1.76
      3        0.92           1.02
      4        0.65           0.75
      5        0.45           0.43
      6        0.40           0.44
      7        0.30           0.36
      8        0.20           0.22
      9        0.16           0.10
     10        0.12           0.64


## What this means for the decision

BG/NBD wins on every point metric, but the decile table is the one that drives spend: its
ranking falls monotonically from 5.09x to 0.12x, while LightGBM's **bottom** decile is worth
more than its middle ones.

With a few thousand customers and a long-tailed target, the structured model generalises
better than the flexible one. That ordering would likely reverse with far more data — the
finding is about this shape of problem, not about the algorithms in general.

Full run: `python -m clv.pipeline`.